In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from pathlib import Path
import uuid
import gc

In [ ]:
encoder = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device="cpu"
)

client = QdrantClient(path="../saves/VScatalogo") # Carrega vectorstore em disco

In [ ]:
file_path = "/home/migueldcarvalho/Downloads/edital_petrobras.pdf"
#file_path = "/home/migueldcarvalho/Downloads/UERJ_2026.pdf"
collection_name = Path(file_path).name

In [ ]:
def embed_texts(texts, encoder, batch_size=5):
    vectors = []
    print(f"Tamanho do texto: {len(texts)}")
    for i in range(0, len(texts), batch_size):
        print(i)
        batch = texts[i:i + batch_size]
        emb = encoder.encode(batch, convert_to_numpy=True)
        vectors.extend(emb)
        del batch, emb
        gc.collect()
    return vectors

def external_doc(state):
    client = state["client"]
    encoder = state["encoder"]


    loader = PyPDFLoader(file_path)
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = text_splitter.split_documents(docs)
    texts = [chunk.page_content for chunk in chunks]

    vectors = embed_texts(texts, encoder, batch_size=5)

    if not client.collection_exists(collection_name=collection_name):
        client.create_collection(
            collection_name=collection_name, # nome da coleção
            vectors_config=models.VectorParams(
            size=encoder.get_sentence_embedding_dimension(),  # Vector size is defined by used model
            distance=models.Distance.COSINE, # Metrica de similaridade
        ),
)

    points = []
    for i, chunk in enumerate(chunks):
        points.append(
            models.PointStruct(
                id=str(uuid.uuid4()),
                vector=vectors[i].tolist(),
                payload={
                    "page_content": chunk.page_content,
                    "metadata": chunk.metadata,
                    "chunk_index": i
                }
            )
        )

    client.upsert(
        collection_name=collection_name,
        points=points
    )

In [ ]:
query = "Qual é o dia de volta as aulas no primeiro periodo?"
query = "Quero informações sobre INSPEÇÃO DE EQUIPAMENTOS E INSTALAÇÕES"

state = {
    "client": client,
    "encoder": encoder
}

if not client.collection_exists(collection_name=collection_name):
        external_doc(state)


In [ ]:
queries = ["Quero informações sobre os requisitos necessário para enfermagem do trabalho",
           "Dados sobre manutenção calderaria",
           "Quero informações sobre INSPEÇÃO DE EQUIPAMENTOS E INSTALAÇÕES",
           "Sobre o que é o edital?",
           "Quais são as datas mais importantes?",
           "Qual é a banca deste concurso?",
           "Quero informações das vagas de PCDs"]

In [ ]:
query = queries[6]

hits = client.query_points(
    collection_name=collection_name,
    query=encoder.encode(query).tolist(),
    limit=5
).points

for hit in hits:
    print(hit.payload['page_content'])
    print()

In [ ]:
p = 8
q = 4
m = 1.2*(p*4) / (32/q)
m